Testing SymPy

In [1]:
import sympy as sp
from sympy.parsing.latex import parse_latex
import random
import constants as C
print(sp.__version__)

1.14.0


In [38]:
x = sp.Symbol('x')
y = sp.Symbol('y')


dy = sp.Symbol('dy', commutative=False)
dx = sp.Symbol('dx', commutative=False)

C1 = sp.Symbol('C1')
dy_dx = sp.Symbol('\\frac{dy}{dx}', commutative=False)

Expression Generation

In [3]:
def get_complex_expr(var, complexity=2):
    """Generates varied mathematical expressions to avoid duplicates."""
    # basics = [var, 1/var, var**2, sp.sin(var), sp.cos(var), sp.tan(var), sp.exp(var), sp.log(var)]
    basics = [var]
    expr = random.choice(basics) * random.randint(1, 10)
    for _ in range(complexity - 1):
        other = random.choice(basics) + random.randint(1, 6)
        op = random.choice(['add', 'mul'])
        expr = expr + other if op == 'add' else expr * other
        print(f"current term : {expr}")
    return sp.simplify(expr)

def get_complex_expr_doubled(var1, var2, complexity=2):
    """Generates varied mathematical expressions to avoid duplicates."""
    vars_basic= [var1, var2, 1/var1, 1/var2]
    vars_extended = vars_basic + [var1**2, var2**2, var1*var2]
    basics_1 = [sp.sin(random.choice(vars_basic)), sp.cos(random.choice(vars_basic)), sp.exp(random.choice(vars_basic))]
    basics = basics_1 + [random.choice(vars_extended)]
    expr = random.choice(basics) * random.randint(1, 10)
    for _ in range(complexity - 1):
        other = random.choice(basics) + random.randint(1, 6)
        op = random.choice(['add', 'mul'])
        expr = expr + other if op == 'add' else expr * other
        print(f"current term : {expr}")
    return sp.simplify(expr)

def generate_separable():
    """Expert for Separable ODEs: dy/dx = f(x)g(y)"""
    complexity = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]
    f_x = get_complex_expr(x, complexity)
    g_y_sym = random.choice([y, y**2, sp.exp(y)])
    ode = sp.Eq(dy_dx, f_x * g_y_sym)
    lhs = sp.integrate(1/g_y_sym, y)
    rhs = sp.integrate(f_x, x)
    eqn = sp.Eq(lhs, rhs + C1)

    if lhs.has(sp.Integral) or rhs.has(sp.Integral):
        return None, None, None, None
    
    steps = [
        {C.STEP: "Classify", C.OP: "Separate Variables", C.RESULT: f"\\frac{{1}}{{{sp.latex(g_y_sym)}}} dy = \\left({sp.latex(f_x)}\\right) dx"},
        {C.STEP: "Integrate LHS", C.OP: "Integrate Left Side", C.RESULT: sp.latex(lhs)},
        {C.STEP: "Integrate RHS", C.OP: "Integrate Right Side", C.RESULT: f"{sp.latex(rhs)} + C_1"},
        {C.STEP: "Solve", C.OP: "Isolate y", C.RESULT: sp.latex(eqn)}
    ]
    soln = sp.solve(eqn, y)
    return "Separable", sp.latex(ode), steps, [sp.latex(s) for s in soln]


Get in Action

In [4]:
complexity = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]
my_func = get_complex_expr(x, complexity=3)
# print(f"The function is : {sp.latex(my_func)} --> || --> {my_func}")

dy_dx = sp.Symbol('\\frac{dy}{dx}')
print(dy_dx)

current term : 5*x*(x + 2)
current term : 5*x*(x + 2)*(x + 5)
\frac{dy}{dx}


In [44]:


eq = '\\frac{dy}{dx} = 9 e^{y} \\log{x}'
sym_eq:sp.Eq = parse_latex(eq)

for der in sym_eq.atoms(sp.Derivative):
    # print("Hello", der, dy/dx)
    sym_eq = sym_eq.subs(der, dy/dx)

# for terms in sym_eq.lhs.
e = sp.Symbol('e')

print(sym_eq)

sym_eq = sp.Eq(sym_eq.lhs*dx/e**y, (sym_eq.rhs*dx/e**y).doit())
print(sym_eq)

Eq(dy*dx**(-1), 9*(e**y*log(x, E)))
Eq(dy/e**y, 9*log(x)*dx)
